# Lesson: Documentation, Knowledge Management, and Project Design


### What you'll do

**1. Hands-on "Auto-generating network documentation"**
- ACL documentation — cryptic rules into plain-English policy
- Topology from CDP — neighbor tables into design insight
- Runbooks from change history — one-time records into reusable procedure

**2. Hands-on "Building a network knowledge base with RAG"**
- Build the knowledge base — chunk and store the design guide in ChromaDB
- The query pipeline — embed, retrieve, ground, respond
- RAG vs. direct LLM — why grounding wins

**3. Hands-on "Designing your first AI-augmented network project"**
- TextFSM + LLM classification — deterministic parsing, intelligent reasoning
- The AI Project Canvas — scope before you code

### Requirements
- Python 3.10+
- A local [Ollama](https://ollama.com) server at `http://127.0.0.1:11434` with `gemma4:e4b` pulled (`ollama pull gemma4:e4b`); no GPU required

### Running scenario
**IntentNet Corp** — the team has a working pipeline; now they capture, organize, and leverage their institutional knowledge.

---

In [1]:
# ============================================================
# Environment Setup
# ============================================================
!pip install -q ollama chromadb textfsm

import json
import re
import textwrap
import ollama

# Configure Ollama client to point at the local server
OLLAMA_HOST = "http://127.0.0.1:11434"
MODEL = "gemma4:e4b"

client = ollama.Client(host=OLLAMA_HOST)

# Reuse the system prompt from the first lesson
NETWORK_SYSTEM_PROMPT = (
    "You are a senior Cisco network engineer with 15 years of experience.\n"
    "You work at IntentNet Corp managing 600 IOS-XE switches across 3 campus locations.\n"
    "When diagnosing issues:\n"
    "- Reference specific Cisco CLI commands\n"
    "- Consider both physical and logical causes\n"
    "- Suggest verification steps before remediation\n"
    "- Flag anything that could cause an outage if done incorrectly\n"
    "Keep responses concise and actionable."
)


# Reload the ask_network_ai() glue helper built in the lesson
# "Bridging NetDevOps and AI" — every LLM call in this notebook goes through it.
def ask_network_ai(
    prompt: str,
    system_prompt: str = NETWORK_SYSTEM_PROMPT,
    json_output: bool = False,
    temperature: float = 0,
) -> str | dict:
    """Send a prompt to the LLM with network engineering context."""
    kwargs = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "options": {"temperature": temperature},
    }
    if json_output:
        kwargs["format"] = "json"

    response = client.chat(**kwargs)
    content = response["message"]["content"]
    return json.loads(content) if json_output else content


print(f"Environment ready. Using {MODEL} at {OLLAMA_HOST}")

Environment ready. Using gemma4:e4b at http://127.0.0.1:11434


---
## 1. Hands-on "Auto-generating network documentation"

47 design guides, 200+ post-mortems — manually written, often outdated. Three parts, one technique:

1. **ACL documentation** — cryptic rules into plain-English policy
2. **Topology from CDP** — neighbor tables into design insight
3. **Runbooks from change history** — one-time records into reusable procedure

> The best documentation is generated, not written. The engineer curates; the AI writes.

### Part 1 — ACL Documentation

IntentNet's edge ACL: six lines of IOS-XE that tell you *what*, never *why*. The prompt asks for plain English per rule, the business intent, a risk note where deserved, and an overall security posture.

In [2]:
# ============================================================
# ACL Documentation Generator
# ============================================================
# IntentNet's edge ACL — real IOS-XE syntax from campus-1 border
edge_acl = """
ip access-list extended EDGE-INBOUND
 10 deny   tcp 10.0.0.0 0.255.255.255 172.16.0.0 0.0.255.255 eq 22
 20 permit tcp 10.0.0.0 0.255.255.255 172.16.0.0 0.0.255.255 eq 443
 30 deny   ip  10.0.0.0 0.255.255.255 192.168.1.0 0.0.0.255
 40 permit icmp any any echo
 50 permit icmp any any echo-reply
 60 deny   ip  any any log
"""

ACL_DOC_PROMPT = """You are a network documentation specialist at IntentNet Corp.

Given an IOS-XE access control list, produce a plain-English documentation page.

Respond ONLY with valid JSON:
{
  "acl_name": "name of the ACL",
  "purpose": "1-2 sentence summary of what this ACL does",
  "rules": [
    {
      "sequence": 10,
      "plain_english": "Human-readable explanation",
      "action": "permit or deny",
      "risk_note": "Any security concern, or null"
    }
  ],
  "security_posture": "Overall assessment of the ACL's security stance",
  "recommendations": ["List of improvements"]
}"""

acl_doc = ask_network_ai(
    f"Document this ACL:\n{edge_acl}",
    system_prompt=ACL_DOC_PROMPT,
    json_output=True,
)

print(f"ACL: {acl_doc['acl_name']}")
print(f"Purpose: {acl_doc['purpose']}")
print(f"\nRule-by-Rule Documentation:")
print("-" * 60)
for rule in acl_doc.get("rules", []):
    print(f"  Seq {rule['sequence']:>3} [{rule['action'].upper():>6}]: {rule['plain_english']}")
    if rule.get("risk_note"):
        print(f"           ⚠ {rule['risk_note']}")
print(f"\nSecurity Posture: {acl_doc['security_posture']}")
print(f"\nRecommendations:")
for rec in acl_doc.get("recommendations", []):
    print(f"  - {rec}")

ACL: EDGE-INBOUND
Purpose: This ACL controls traffic entering the network edge, specifically managing SSH access and allowing HTTPS while blocking unauthorized internal IP ranges.

Rule-by-Rule Documentation:
------------------------------------------------------------
  Seq  10 [  DENY]: Deny all incoming TCP connections destined for port 22 (SSH) originating from the entire 10.0.0.0/8 range and targeting the 172.16.0.0/16 subnet.
           ⚠ This rule appears overly specific and might block legitimate SSH access if the source or destination ranges are incorrect.
  Seq  20 [PERMIT]: Permit all incoming TCP connections destined for port 443 (HTTPS) originating from the entire 10.0.0.0/8 range and targeting the 172.16.0.0/16 subnet.
  Seq  30 [  DENY]: Deny all IP traffic originating from the entire 10.0.0.0/8 range and targeting the specific host 192.168.1.1.
  Seq  40 [PERMIT]: Permit all incoming ICMP echo requests (pings) from any source to any destination.
           ⚠ Allowing ge

Five seconds per ACL, every rule documented, security analysis included — versus thirty minutes by hand, with the "obvious" rules skipped.

---
### Part 2 — Topology from CDP

The neighbor table already describes your physical topology. Feed it in and ask for roles, link descriptions, and design observations — insights, not just a summary.

In [4]:
# ============================================================
# CDP Neighbor Topology Summary
# ============================================================
# CDP neighbor table from IntentNet's core-rtr-01
cdp_neighbors = """
Device ID        Local Intrfce     Holdtme    Capability  Platform  Port ID
dist-sw-21       Gig 0/1           145             R S I  WS-C3850  Gig 1/0/1
dist-sw-22       Gig 0/2           160             R S I  WS-C3850  Gig 1/0/1
core-rtr-02      Gig 0/3           172               R    ISR4451   Gig 0/3
wan-rtr-01       Gig 0/4           138               R    ASR1001   Gig 0/0/0
"""

CDP_DOC_PROMPT = """You are a network documentation specialist at IntentNet Corp.

Given a CDP neighbor table, produce a structured topology summary.

Respond ONLY with valid JSON:
{
  "source_device": "device the CDP table was collected from",
  "neighbors": [
    {
      "device": "neighbor hostname",
      "role": "core / distribution / access / WAN",
      "platform": "platform model",
      "link_description": "Plain-English description of the link",
      "potential_concerns": "Any design concern, or null"
    }
  ],
  "topology_summary": "2-3 sentence description of the topology around this device",
  "design_observations": ["Noteworthy observations about the topology design"]
}"""

topo_doc = ask_network_ai(
    f"Generate topology summary from this CDP table collected from core-rtr-01:\n{cdp_neighbors}",
    system_prompt=CDP_DOC_PROMPT,
    json_output=True,
)

print(f"Source Device: {topo_doc['source_device']}")
print(f"\nTopology Summary: {topo_doc['topology_summary']}")
print(f"\nNeighbor Details:")
print("-" * 60)
for nbr in topo_doc.get("neighbors", []):
    print(f"  {nbr['device']} ({nbr['role']}) — {nbr['platform']}")
    print(f"    Link: {nbr['link_description']}")
    if nbr.get("potential_concerns"):
        print(f"    ⚠ {nbr['potential_concerns']}")
print(f"\nDesign Observations:")
for obs in topo_doc.get("design_observations", []):
    print(f"  - {obs}")

Source Device: core-rtr-01

Topology Summary: core-rtr-01 acts as a central aggregation point, connecting multiple critical network segments. It maintains links to two distribution switches, one core router, and the primary WAN edge device. This setup suggests a robust, multi-layered design supporting both internal campus connectivity and external internet access.

Neighbor Details:
------------------------------------------------------------
  dist-sw-21 (distribution) — WS-C3850
    Link: Link to distribution switch 21 (Gigabit Ethernet)
  dist-sw-22 (distribution) — WS-C3850
    Link: Link to distribution switch 22 (Gigabit Ethernet)
  core-rtr-02 (core) — ISR4451
    Link: Link to core router 2 (Gigabit Ethernet)
  wan-rtr-01 (WAN) — ASR1001
    Link: Link to WAN router 1 (Gigabit Ethernet)

Design Observations:
  - The presence of dedicated connections to separate core (core-rtr-02) and WAN (wan-rtr-01) devices indicates proper segmentation of network functions.
  - Connecting mul

---
### Part 3 — Runbooks from Change History

Sarah Chen's OSPF area migration was a one-time change record. The engineer moves on; the runbook stays: prerequisites, numbered steps with CLI and expected results, a validation checklist, and a rollback plan.

In [6]:
# ============================================================
# Runbook Generation from Change History
# ============================================================
# A one-time change record from IntentNet's change management system
change_history = """
Change #CR-2024-0147: OSPF Area 0 migration on campus-2
Date: 2024-01-15
Engineer: Sarah Chen
Steps taken:
1. Verified OSPF neighbor states on dist-sw-21 and dist-sw-22
2. Added new area 0 network statements under router ospf 1
3. Removed old area 10 network statements
4. Verified convergence — all neighbors reached FULL state
5. Confirmed reachability to all campus-2 subnets from core
Rollback plan: Restore area 10 statements from backup config
Duration: 25 minutes (within change window)
"""

RUNBOOK_PROMPT = """You are a network documentation specialist at IntentNet Corp.

Given a completed change record, generate a reusable runbook template that
any engineer could follow for similar changes in the future.

Respond ONLY with valid JSON:
{
  "runbook_title": "Descriptive title",
  "scope": "What this runbook covers",
  "prerequisites": ["List of items to verify before starting"],
  "steps": [
    {
      "step": 1,
      "action": "What to do",
      "cli_commands": ["IOS-XE commands to run"],
      "expected_result": "What you should see",
      "rollback": "How to undo this step if needed"
    }
  ],
  "validation_checklist": ["Post-change verification items"],
  "estimated_duration": "Time estimate",
  "risk_level": "low / medium / high",
  "rollback_plan": "Full rollback procedure"
}"""

runbook = ask_network_ai(
    f"Generate a reusable runbook from this change record:\n{change_history}",
    system_prompt=RUNBOOK_PROMPT,
    json_output=True,
)

print(f"RUNBOOK: {runbook['runbook_title']}")
print(f"Scope: {runbook['scope']}")
print(f"Risk Level: {runbook['risk_level']}")
print(f"Estimated Duration: {runbook['estimated_duration']}")
print(f"\nPrerequisites:")
for prereq in runbook.get("prerequisites", []):
    print(f"  [ ] {prereq}")
print(f"\nSteps:")
print("-" * 60)
for step in runbook.get("steps", []):
    print(f"  Step {step['step']}: {step['action']}")
    for cmd in step.get("cli_commands", []):
        print(f"    $ {cmd}")
    print(f"    Expected: {step['expected_result']}")
print(f"\nValidation Checklist:")
for item in runbook.get("validation_checklist", []):
    print(f"  [ ] {item}")
print(f"\nRollback Plan: {runbook['rollback_plan']}")

RUNBOOK: Migrate OSPF Area 0 Network Statements on Campus Distribution Switches
Scope: This runbook details the procedure for migrating network statements from an old OSPF area (e.g., Area 10) to the backbone Area 0 on distribution switches within a campus environment.
Risk Level: medium
Estimated Duration: 25 minutes

Prerequisites:
  [ ] Access credentials and appropriate permissions for the target distribution switches (dist-sw-21, dist-sw-22).
  [ ] A current backup configuration of the affected routers.
  [ ] Knowledge of the IP addressing scheme and subnet ranges to be migrated.

Steps:
------------------------------------------------------------
  Step 1: Verify Current OSPF Neighbor Status
    $ show ip ospf neighbor detail
    $ show ip route ospf area 0
    Expected: All critical neighbors should be in the FULL state. Verify that existing Area 0 routes are stable.
  Step 2: Add New Network Statements to OSPF Process (Area 0)
    $ configure terminal
    $ router ospf 1
    $ 

---
## 2. Hands-on "Building a network knowledge base with RAG"

RAG is DNS for your documentation: DNS resolves a name to an IP; RAG resolves *"What's our standard WAN MTU?"* to the answer in your design guide. Three parts:

1. **Build the knowledge base** — chunk and store the design guide
2. **The query pipeline** — embed, retrieve, ground, respond
3. **RAG vs. direct LLM** — why grounding wins

The pipeline: **Ingest → Chunk → Embed → Store → Query → Ground → Respond**.

### Part 1 — Build the Knowledge Base

The design guide is chunked by section headers; ChromaDB stores the chunks and embeds them with its built-in local model — everything stays on your machine.

In [7]:
# ============================================================
# Prepare the Knowledge Base — Network Design Guide
# ============================================================
import chromadb

# IntentNet's network design guide (simulated excerpts)
design_guide = """
IntentNet Corp — Network Design Guide v4.2

## WAN Standards
- All WAN interfaces use MTU 9000 (jumbo frames) for ISP-A peering
- WAN interfaces to ISP-B use MTU 1500 (standard) per their peering agreement
- BGP keepalive timer: 60 seconds, hold time: 180 seconds
- All WAN links use BFD with 300ms detect interval

## Campus Standards
- Access layer: 48-port Catalyst 3850 switches
- Distribution layer: Catalyst 9300 with StackWise Virtual
- OSPF area design: Area 0 for backbone, one area per campus building
- STP: Rapid PVST+ with root bridge on distribution switches
- MTU: 1500 for all campus interfaces (no jumbo frames on access layer)

## Security Standards
- All management access via SSH v2 only
- TACACS+ primary authentication, local fallback
- SNMPv3 authPriv only — no v2c community strings
- Unused ports: shutdown and assigned to VLAN 999 (blackhole)

## Monitoring Standards
- Syslog to 10.1.100.50 (primary) and 10.1.100.51 (backup)
- SNMP polling interval: 300 seconds
- Streaming telemetry via gNMI for core and distribution devices
- NetFlow v9 on all WAN interfaces for traffic analysis
"""

# ---- Step 1: Chunk the document by section headers ----
sections = re.split(r"(?=^## )", design_guide.strip(), flags=re.MULTILINE)
# Keep only the "## " section chunks — the guide's title line is not content
sections = [s.strip() for s in sections if s.strip().startswith("## ")]

print(f"Document chunked into {len(sections)} sections:")
for i, section in enumerate(sections):
    # Extract the first line as the section title
    title = section.split("\n")[0].replace("## ", "").strip()
    print(f"  Chunk {i}: {title} ({len(section)} chars)")

# ---- Steps 2 + 3: Embed and store in ChromaDB ----
# ChromaDB embeds each chunk with its built-in local sentence-transformer
# model (all-MiniLM-L6-v2) -- no external API needed.
chroma_client = chromadb.Client()  # In-memory for this demo
collection = chroma_client.create_collection(
    name="intentnet_design_guide",
    metadata={"hnsw:space": "cosine"},
)

# Add chunks -- ChromaDB embeds them on insert
collection.add(
    ids=[f"chunk_{i}" for i in range(len(sections))],
    documents=sections,
    metadatas=[{"source": "design_guide_v4.2", "chunk_index": i} for i in range(len(sections))],
)

print(f"\nStored and embedded {collection.count()} chunks in ChromaDB.")
print("Knowledge base ready for queries.")

Document chunked into 4 sections:
  Chunk 0: WAN Standards (270 chars)
  Chunk 1: Campus Standards (328 chars)
  Chunk 2: Security Standards (222 chars)
  Chunk 3: Monitoring Standards (240 chars)

Stored and embedded 4 chunks in ChromaDB.
Knowledge base ready for queries.


---
### Part 2 — The Query Pipeline

`query_knowledge_base()`: the question is embedded, ChromaDB returns the nearest chunks, the chunks become the LLM's context — and the grounding prompt forbids answering from outside it, with a citation of the section used.

In [8]:
# ============================================================
# Build the RAG Query Pipeline
# ============================================================
def query_knowledge_base(question: str, collection, n_results: int = 2) -> str:
    """Query the knowledge base and generate a grounded response."""

    # Steps 1 + 2: Embed the question and search ChromaDB
    results = collection.query(
        query_texts=[question],
        n_results=n_results,
    )

    # Step 3: Build context from retrieved chunks
    retrieved_docs = results["documents"][0]
    context = "\n\n---\n\n".join(retrieved_docs)

    # Step 4: Send to LLM with grounding instructions
    RAG_PROMPT = (
        "You are a network knowledge base assistant for IntentNet Corp.\n"
        "Answer the question using ONLY the provided context from the design guide.\n"
        "If the context does not contain the answer, say: "
        "'This is not covered in the current design guide.'\n"
        "Always cite which section of the design guide your answer comes from."
    )

    return ask_network_ai(
        f"CONTEXT FROM DESIGN GUIDE:\n{context}\n\nQUESTION: {question}",
        system_prompt=RAG_PROMPT,
    )


print("RAG query function defined. Ready to test.")

RAG query function defined. Ready to test.


In [9]:
# ============================================================
# Test the RAG Pipeline
# ============================================================
# Four real questions an IntentNet engineer might ask
questions = [
    "What's our standard MTU for WAN interfaces?",
    "Which SNMP version do we use?",
    "How is OSPF area design structured across campuses?",
    "What's the BFD detect interval for WAN links?",
]

print("RAG Pipeline Test Results")
print("=" * 60)
for q in questions:
    answer = query_knowledge_base(q, collection)
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print("-" * 60)

RAG Pipeline Test Results

Q: What's our standard MTU for WAN interfaces?
A: The MTU for WAN interfaces depends on the peering partner:

*   For ISP-A peering, all WAN interfaces use an MTU of 9000 (jumbo frames) (WAN Standards).
*   For WAN interfaces to ISP-B, the MTU is 1500 (standard), per their peering agreement (WAN Standards).
------------------------------------------------------------

Q: Which SNMP version do we use?
A: We must use SNMPv3 with `authPriv` only, and v2c community strings are prohibited (Security Standards).
------------------------------------------------------------

Q: How is OSPF area design structured across campuses?
A: OSPF utilizes Area 0 for the backbone, with a separate area designated for each campus building (Campus Standards).

(Source: Campus Standards)
------------------------------------------------------------

Q: What's the BFD detect interval for WAN links?
A: All WAN links use a BFD detect interval of 300ms (WAN Standards).
------------------

Four real engineer questions, four grounded answers, four citations — all from a guide the model has never seen in training.

---
### Part 3 — RAG vs. Direct LLM

Same question, with and without grounding: *"What MTU should I configure on the WAN interface to ISP-B?"*

> No foundation model knows that IntentNet's ISP-B peering agreement requires MTU 1500. Only your documents know.

In [10]:
# ============================================================
# RAG vs. Direct LLM Comparison
# ============================================================
# This question has an IntentNet-specific answer the LLM cannot know
question = "What MTU should I configure on the WAN interface to ISP-B?"

# ----- Without RAG: LLM guesses (likely wrong or generic) -----
answer_no_rag = ask_network_ai(question)

# ----- With RAG: LLM answers from the design guide -----
answer_with_rag = query_knowledge_base(question, collection)

print("Question:", question)
print("\n" + "=" * 60)
print("WITHOUT RAG (direct LLM):")
print("-" * 60)
print(textwrap.fill(answer_no_rag, width=80))
print("\n" + "=" * 60)
print("WITH RAG (grounded in design guide):")
print("-" * 60)
print(textwrap.fill(answer_with_rag, width=80))
print("\n" + "=" * 60)
print("\nKey difference: The RAG answer references IntentNet's specific")
print("ISP-B peering agreement (MTU 1500), while the direct LLM")
print("gives a generic answer without organizational context.")

Question: What MTU should I configure on the WAN interface to ISP-B?

WITHOUT RAG (direct LLM):
------------------------------------------------------------
**Do not configure an MTU value without explicit confirmation from ISP-B.**
Incorrectly setting the MTU is a primary cause of intermittent connectivity,
packet loss, and application failure (often manifesting as "random drops" or
slow performance).  The correct MTU depends entirely on the encapsulation method
used by ISP-B.  ### 1. Verification Steps (Before Remediation)  **A. Consult
Documentation:** *   **Action:** Obtain the official Service Level Agreement
(SLA) or technical guide from ISP-B. They will specify the required MTU and any
necessary overhead adjustments (e.g., PPPoE adds significant overhead). *
**Goal:** Determine if they require standard Ethernet (1500 bytes), a smaller
value due to tunneling, or a larger jumbo frame size.  **B. Test Path Discovery
(The Gold Standard):** *   Use the `ping` command with the "Don't 

---
## 3. Hands-on "Designing your first AI-augmented network project"

You've seen the tools — now design a real project, in two parts:

1. **TextFSM + LLM classification** — deterministic parsing feeding intelligent reasoning
2. **The AI Project Canvas** — scope before you code

> Design your AI project the way you design a routing domain. Bounded. Purposeful. Testable.

### Part 1 — TextFSM + LLM Classification

One TextFSM template — six fields — turns raw syslog into structured dictionaries; the LLM then classifies category and urgency and summarizes the situation. Deterministic steps first, expensive AI steps after.

In [11]:
# ============================================================
# Data Preparation — Syslog Parsing with TextFSM
# ============================================================
# One TextFSM template, thousands of messages parsed.
import io
import textfsm

SYSLOG_TEMPLATE = r"""Value TIMESTAMP (\w+ \d+ [\d:]+)
Value HOSTNAME (\S+)
Value FACILITY (\w+)
Value SEVERITY (\d)
Value MNEMONIC (\w+)
Value MESSAGE (.+)

Start
  ^${TIMESTAMP} ${HOSTNAME} %${FACILITY}-${SEVERITY}-${MNEMONIC}: ${MESSAGE} -> Record
"""

raw_syslog = """
Feb 14 03:22:15 core-rtr-02 %OSPF-5-ADJCHG: Process 1, Nbr 10.1.1.2 on Gi0/1 from FULL to DOWN, Neighbor Down: Dead timer expired
Feb 14 03:22:17 core-rtr-02 %LINK-3-UPDOWN: Interface GigabitEthernet0/1, changed state to down
Feb 14 03:22:18 dist-sw-21 %STP-4-ROLE_FWD: Port Po1 instance 100 role changed from alternate to designated
Feb 14 03:22:20 acc-sw-201 %CDP-4-NATIVE_VLAN_MISMATCH: Native VLAN mismatch on Gi1/0/1 (201), with dist-sw-21 Gi1/0/48 (1)
Feb 14 03:22:22 core-rtr-02 %SYS-5-RESTART: System restarted
Feb 14 03:28:45 core-rtr-02 %LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet0/1, changed state to up
"""

fsm = textfsm.TextFSM(io.StringIO(SYSLOG_TEMPLATE))
rows = fsm.ParseText(raw_syslog.strip())
parsed_logs = [dict(zip([h.lower() for h in fsm.header], row)) for row in rows]

print(f"Parsed {len(parsed_logs)} syslog entries into structured format:")
print(f"Fields: {list(parsed_logs[0].keys())}")
print()
for log in parsed_logs:
    print(f"  [{log['severity']}] {log['hostname']} | {log['facility']}-{log['mnemonic']}: {log['message'][:60]}...")

# ---- Feed structured data to LLM for classification ----
CLASSIFY_PROMPT = """You are a syslog analyst for IntentNet Corp.

Given structured syslog entries, classify each into an operational category
and assess the overall situation.

Respond ONLY with valid JSON:
{
  "classifications": [
    {
      "hostname": "device",
      "mnemonic": "the mnemonic",
      "category": "routing / switching / security / system / interface",
      "urgency": "critical / warning / informational"
    }
  ],
  "situation_summary": "1-2 sentence summary of what happened",
  "recommended_investigation": ["ordered list of next steps"]
}"""

classification = ask_network_ai(
    f"Classify these parsed syslog entries:\n{json.dumps(parsed_logs, indent=2)}",
    system_prompt=CLASSIFY_PROMPT,
    json_output=True,
)

print(f"\nSituation: {classification['situation_summary']}")
print(f"\nClassifications:")
for c in classification.get("classifications", []):
    print(f"  {c['hostname']:>15} | {c['mnemonic']:<20} | {c['category']:<12} | {c['urgency']}")
print(f"\nRecommended Investigation:")
for step in classification.get("recommended_investigation", []):
    print(f"  - {step}")

Parsed 6 syslog entries into structured format:
Fields: ['timestamp', 'hostname', 'facility', 'severity', 'mnemonic', 'message']

  [5] core-rtr-02 | OSPF-ADJCHG: Process 1, Nbr 10.1.1.2 on Gi0/1 from FULL to DOWN, Neighbor...
  [3] core-rtr-02 | LINK-UPDOWN: Interface GigabitEthernet0/1, changed state to down...
  [4] dist-sw-21 | STP-ROLE_FWD: Port Po1 instance 100 role changed from alternate to designa...
  [4] acc-sw-201 | CDP-NATIVE_VLAN_MISMATCH: Native VLAN mismatch on Gi1/0/1 (201), with dist-sw-21 Gi1/0...
  [5] core-rtr-02 | SYS-RESTART: System restarted...
  [5] core-rtr-02 | LINEPROTO-UPDOWN: Line protocol on Interface GigabitEthernet0/1, changed state...

Situation: The core router experienced a link failure and subsequent system restart, followed by the interface recovering its line protocol. Separately, an access switch reported a native VLAN mismatch with a distribution switch.

Classifications:
      core-rtr-02 | ADJCHG               | routing      | warning
      cor

TextFSM parses; the LLM reasons. Each tool doing what it does best — and the parse step is free, fast, and exactly reproducible.

---
### Part 2 — The AI Project Canvas

Feed a project description, get the full scoping document: problem statement, scope, data requirements, approach, success metrics, risks, and a phased timeline. The `technical_approach` section makes the RAG vs. fine-tuning vs. prompt-engineering call from the slides — when in doubt, start with RAG; you can always fine-tune later. Complete the canvas before you open a notebook — a project without scope is a wish.

In [12]:
# ============================================================
# AI Project Canvas Generator
# ============================================================
# Scope a real project using a structured canvas
project_description = (
    "Build an AI assistant that answers questions about IntentNet's "
    "network design standards and past incident post-mortems. "
    "The assistant should be accessible via a Webex bot and "
    "grounded in our internal documentation."
)

CANVAS_PROMPT = """You are an AI project planning advisor for network engineering teams.

Given a project description, generate a structured project canvas that covers
all critical planning dimensions.

Respond ONLY with valid JSON:
{
  "project_name": "Short, descriptive name",
  "problem_statement": "What specific problem does this solve?",
  "scope": {
    "in_scope": ["What the project WILL do"],
    "out_of_scope": ["What the project will NOT do (yet)"]
  },
  "data_requirements": [
    {
      "data_source": "Where the data comes from",
      "format": "Data format",
      "volume": "Estimated volume",
      "refresh_frequency": "How often it updates"
    }
  ],
  "technical_approach": {
    "primary_technique": "RAG / fine-tuning / prompt-engineering",
    "model": "Recommended model",
    "infrastructure": "Where it runs",
    "key_components": ["List of components to build"]
  },
  "success_metrics": [
    {"metric": "What to measure", "target": "Target value", "measurement_method": "How to measure"}
  ],
  "risks": [
    {"risk": "What could go wrong", "mitigation": "How to address it", "likelihood": "low/medium/high"}
  ],
  "timeline": [
    {"phase": "Phase name", "duration": "Time estimate", "deliverable": "What's produced"}
  ],
  "estimated_monthly_cost": "Rough cost estimate"
}"""

canvas = ask_network_ai(
    f"Generate a project canvas for:\n{project_description}",
    system_prompt=CANVAS_PROMPT,
    json_output=True,
)

print(f"AI PROJECT CANVAS: {canvas.get('project_name', 'N/A')}")
print("=" * 60)
print(f"\nProblem: {canvas.get('problem_statement', 'N/A')}")

print(f"\nScope:")
print(f"  In scope:")
for item in canvas.get("scope", {}).get("in_scope", []):
    print(f"    + {item}")
print(f"  Out of scope:")
for item in canvas.get("scope", {}).get("out_of_scope", []):
    print(f"    - {item}")

print(f"\nTechnical Approach:")
tech = canvas.get("technical_approach", {})
print(f"  Technique: {tech.get('primary_technique', '?')}")
print(f"  Model: {tech.get('model', '?')}")
print(f"  Infrastructure: {tech.get('infrastructure', '?')}")
print(f"  Components: {', '.join(tech.get('key_components', []))}")

print(f"\nSuccess Metrics:")
for m in canvas.get("success_metrics", []):
    print(f"  - {m.get('metric', '?')}: {m.get('target', '?')}")

print(f"\nTimeline:")
for phase in canvas.get("timeline", []):
    print(f"  {phase.get('phase', '?')} ({phase.get('duration', '?')}): {phase.get('deliverable', '?')}")

print(f"\nRisks:")
for r in canvas.get("risks", []):
    print(f"  [{r.get('likelihood', '?').upper()}] {r.get('risk', '?')}")
    print(f"    Mitigation: {r.get('mitigation', '?')}")

print(f"\nEstimated Monthly Cost: {canvas.get('estimated_monthly_cost', 'N/A')}")

AI PROJECT CANVAS: IntentNet Knowledge Assistant Bot

Problem: Network engineers spend excessive time manually searching through disparate, complex, and voluminous internal documents (design standards, post-mortems) to find accurate answers regarding network design requirements or root causes of past incidents.

Scope:
  In scope:
    + Developing an AI assistant capable of answering natural language questions about IntentNet's network design standards.
    + Integrating the assistant into a Webex bot interface for accessibility.
    + Grounding all answers strictly in provided internal documentation (RAG implementation).
    + Handling queries related to past incident post-mortems and their findings.
  Out of scope:
    - Modifying or updating the source documentation itself.
    - Providing real-time network diagnostic capabilities or executing commands.
    - Supporting multiple communication channels beyond Webex (e.g., Slack, Teams).
    - Handling subjective or speculative questi

---
## Lesson Summary

You:

1. **Auto-generated documentation** from ACLs, CDP tables, and change history
2. **Built a RAG pipeline** with ChromaDB and queried the design guide in natural language
3. **Compared RAG vs. direct LLM** — grounding wins for organization-specific knowledge
4. **Prepared data and scoped a project** — TextFSM parsing and the AI Project Canvas

**Key insight:** your network generates knowledge every day — configs, logs, incidents, changes. RAG turns it into an always-available assistant. Like DNS for your documentation.